# Hierarchical RAG — Implementation Template

## What is Hierarchical RAG?
Standard RAG searches all **889 chunks** with a single query — every function
competes equally regardless of relevance.

Hierarchical RAG narrows the search space with a two-pass approach:

```
Layer 1 (coarse)  ~50 file-level summary vectors
                  → Fast. Finds the right MODULE.

Layer 2 (fine)    ~889 function-level chunk vectors  (already in Pinecone)
                  → Only searched within files that matched Layer 1.
```

For FastAPI this matters:
- `"OAuth2 token validation"` should only search `security/` files, not `routing/`
- Smaller candidate pool → `is_union_of_base_models` stops appearing in results

## Your tasks
| Step | Status | What to do |
|---|---|---|
| 0 · Install | ready | Run once |
| 1 · Load | ready | Run as-is |
| 2 · Summary index | ready | Run as-is |
| 3 · HierarchicalRAG | **partial** | Choose Option A or B for Pass 2 |
| 4 · Quick test | ready | Plug in your Pinecone key and run |
| 5 · Tune TOP_FILES | ready | Try 3, 5, 7 and compare |
| 6 · Eval | ready | Adds to `chunking_test.ipynb` automatically |

## 0 · Install

In [ ]:
%pip install -q pinecone-client sentence-transformers rank_bm25

## 1 · Imports and config

In [ ]:
import os, re, json
from pathlib import Path
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone

import rag_pipeline, importlib
importlib.reload(rag_pipeline)
from rag_pipeline import VersionControlRAG

PINECONE_KEY = "YOUR_PINECONE_API_KEY"   # ← paste your key
INDEX_NAME   = "python-rag-v3"
NAMESPACE    = "summaries"               # separate from chunk vectors
DATA_DIR     = Path(".")

with open(DATA_DIR / "local_corpus.json") as f:
    corpus = json.load(f)

print(f"Corpus: {len(corpus)} chunks")

## 2 · Build the file-level summary index

Creates one summary vector per **source file** and upserts them into
Pinecone under the `summaries` namespace.

**Runs automatically — no changes needed.**  
Skips itself if vectors already exist.

In [ ]:
def build_summary_index(corpus, pinecone_client, embedder,
                        index_name=INDEX_NAME, namespace=NAMESPACE):
    index = pinecone_client.Index(index_name)

    # Skip if already built
    try:
        ns_stats   = index.describe_index_stats().get("namespaces", {})
        n_existing = ns_stats.get(namespace, {}).get("vector_count", 0)
        if n_existing > 0:
            print(f"Summary index already has {n_existing} vectors — skipping.")
            return
    except Exception:
        pass

    def source_file(doc_id):  return re.sub(r"_f\d+$", "", doc_id)
    def has_chinese(t):       return any("\u4e00" <= c <= "\u9fff" for c in t)

    # Group chunks by source file
    by_file = defaultdict(list)
    for item in corpus:
        by_file[source_file(item["id"])].append(item)

    # Build one summary text per file
    upsert_batch = []
    for file_id, chunks in by_file.items():
        parts = []
        for chunk in chunks:
            summary = chunk["metadata"].get("summary", "")
            if summary and not has_chinese(summary):
                clean = re.sub(r"^\[[^\]]+\]\s*", "", summary).strip()
                if clean:
                    parts.append(clean)
            first_line = chunk["content"].strip().splitlines()[0]
            if first_line.startswith(("def ", "async def ", "class ")):
                parts.append(first_line)

        if not parts:
            parts = [chunks[0]["content"][:200]]

        summary_text = " | ".join(parts[:20])
        tier         = chunks[0]["metadata"].get("tier", "docs_src")

        upsert_batch.append({
            "id":     file_id,
            "values": embedder.encode(summary_text).tolist(),
            "metadata": {
                "file_id":      file_id,
                "tier":         tier,
                "n_chunks":     len(chunks),
                "summary_text": summary_text[:500],
            }
        })

    for i in range(0, len(upsert_batch), 100):
        index.upsert(vectors=upsert_batch[i:i+100], namespace=namespace)

    print(f"✅ Summary index: {len(upsert_batch)} file-level vectors in '{namespace}'")


print("build_summary_index defined — will run in Step 4 when RAG is loaded.")

## 3 · HierarchicalRAG class

### Pass 1 (implemented)
Query the `summaries` namespace → returns top-N file IDs.

### Pass 2 — YOUR TASK
Search only the chunks from those files.

**Option A — simple, works immediately:**  
Filter `local_corpus` to candidate chunk IDs, then rerank with the cross-encoder.
No changes to `rag_pipeline_fixed.py` needed.

**Option B — Pinecone-native, faster at scale:**  
Query Pinecone with a `{"file_id": {"$in": top_file_ids}}` filter.  
Requires adding `"file_id"` to chunk metadata in `rag_pipeline_fixed.ingest_data()`.

The current code uses **Option A** as the default.  
Uncomment the Option B block to switch.

In [ ]:
class HierarchicalRAG:
    TOP_FILES = 3   # ← tune this: try 3, 5, 7 (see Step 5)

    def __init__(self, rag_instance: VersionControlRAG,
                 corpus_path: str = "local_corpus.json",
                 namespace: str   = NAMESPACE):
        self.rag       = rag_instance
        self.namespace = namespace

        with open(corpus_path) as f:
            corp = json.load(f)

        def source_file(doc_id): return re.sub(r"_f\d+$", "", doc_id)

        self.file_to_chunk_ids = defaultdict(list)
        self.id_to_content     = {}
        for item in corp:
            self.file_to_chunk_ids[source_file(item["id"])].append(item["id"])
            self.id_to_content[item["id"]] = item["content"]

        # Build summary index (skips if already exists)
        build_summary_index(corp, rag_instance.pc, rag_instance.embedder)
        print("HierarchicalRAG ready.")

    def _pass1_find_files(self, query_vector: list[float]) -> list[str]:
        """Query summary namespace → return top file IDs."""
        results = self.rag.index.query(
            vector           = query_vector,
            top_k            = self.TOP_FILES,
            namespace        = self.namespace,
            include_metadata = True,
        )
        return [m["id"] for m in results["matches"]]

    def retrieve_hierarchical(self, query: str, top_k: int = 5,
                               use_hyde: bool = False) -> list[str]:
        """Two-pass retrieval. Drop-in for retrieve_complex()."""

        # Embed query (or hypothesis if use_hyde=True)
        if use_hyde and not self.rag.ingest_only:
            hyp          = self.rag._generate_hypothesis(query)
            query_vector = self.rag.embedder.encode(hyp).tolist()
        else:
            query_vector = self.rag.embedder.encode(query).tolist()

        # ── Pass 1 ───────────────────────────────────────────────────────────
        top_file_ids = self._pass1_find_files(query_vector)
        print(f"[Hierarchical] Pass 1 → {len(top_file_ids)} files selected")
        for fid in top_file_ids:
            print(f"  {fid.split('_')[-2] if '_' in fid else fid}")

        if not top_file_ids:
            print("  ⚠ No files found — falling back to full search")
            return self.rag.retrieve_complex(query, top_k=top_k, mode="advanced")

        # ── Pass 2: Option A (default — works with no pipeline changes) ───────
        candidate_ids     = [cid for fid in top_file_ids
                              for cid in self.file_to_chunk_ids.get(fid, [])]
        candidate_content = [self.id_to_content[cid]
                              for cid in candidate_ids
                              if cid in self.id_to_content]
        print(f"[Hierarchical] Pass 2 → {len(candidate_content)} candidate chunks")

        if not candidate_content:
            return self.rag.retrieve_complex(query, top_k=top_k, mode="advanced")

        pairs  = [[query, code] for code in candidate_content]
        scores = self.rag.reranker.predict(pairs)
        ranked = sorted(zip(candidate_content, scores),
                        key=lambda x: x[1], reverse=True)
        return [code for code, _ in ranked[:top_k]]

        # ── Pass 2: Option B — uncomment to use Pinecone-native filter ────────
        # Requires: metadata["file_id"] = source_file(doc_id) in ingest_data()
        #
        # pinecone_res = self.rag.index.query(
        #     vector           = query_vector,
        #     filter           = {"file_id": {"$in": top_file_ids}},
        #     top_k            = max(top_k * 3, 15),
        #     include_metadata = True,
        # )
        # candidates = [m["metadata"]["code"] for m in pinecone_res["matches"]]
        # if not candidates:
        #     return self.rag.retrieve_complex(query, top_k=top_k, mode="advanced")
        # pairs  = [[query, code] for code in candidates]
        # scores = self.rag.reranker.predict(pairs)
        # ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
        # return [code for code, _ in ranked[:top_k]]


print("✅ HierarchicalRAG class defined")

## 4 · Load RAG and run a quick test

In [ ]:
rag = VersionControlRAG(
    pinecone_key = PINECONE_KEY,
    model_path   = "Qwen/Qwen2.5-Coder-7B",
    adapter_path = "Ivan17Ji/qwen-lora-250",
    ingest_only  = False,
)
rag.load_local_corpus(str(DATA_DIR / "local_corpus.json"))

In [ ]:
hier = HierarchicalRAG(
    rag_instance = rag,
    corpus_path  = str(DATA_DIR / "local_corpus.json"),
)

query = "How does OAuth2 password flow work in FastAPI?"
print(f"Query: {query}\n")

baseline = rag.retrieve_complex(query, top_k=3, mode="advanced")
hier_res = hier.retrieve_hierarchical(query, top_k=3)

print("\nBaseline:")
for i, r in enumerate(baseline):
    fn = re.search(r"(?:async )?def (\w+)", r)
    print(f"  [{i}] {fn.group(1) if fn else '?'}")

print("\nHierarchical:")
for i, r in enumerate(hier_res):
    fn = re.search(r"(?:async )?def (\w+)", r)
    print(f"  [{i}] {fn.group(1) if fn else '?'}")

## 5 · Tune TOP_FILES

Higher `TOP_FILES` → more recall, less precision.  
Lower `TOP_FILES` → better precision, risk of missing the right file.

Run this cell to compare three settings side by side.

In [ ]:
import json, re, random
from pathlib import Path

with open(DATA_DIR / "ragas_dataset.jsonl") as f:
    ragas_records = [json.loads(l) for l in f]
with open(DATA_DIR / "fastapi_golden_set_splits.json") as f:
    splits = json.load(f)

test_q   = {x["instruction"] for x in splits["test"]}
test_rec = [r for r in ragas_records if r["question"] in test_q]
random.seed(42)
sample   = random.sample(test_rec, 15)

def semantic_hit(retrieved, golden):
    golden_fns = set(re.findall(r"(?:async )?def (\w+)", " ".join(golden)))
    golden_fns -= {"__init__", "__call__", "main"}
    ret_fns    = set(re.findall(r"(?:async )?def (\w+)", " ".join(retrieved)))
    return bool(golden_fns & ret_fns) if golden_fns else False

print(f"{'TOP_FILES':>10}  {'hit_rate':>8}")
print("-" * 22)
for n_files in [3, 5, 7]:
    hier.TOP_FILES = n_files
    hits = sum(
        1 for r in sample
        if semantic_hit(hier.retrieve_hierarchical(r["question"], top_k=5), r["contexts"])
    )
    print(f"{n_files:>10}  {hits/len(sample):>8.1%}")

## 6 · Add to chunking_test.ipynb

In `chunking_test.ipynb` **Section 3** (Build retrievers), add:

```python
# ── Hierarchical RAG retriever ────────────────────────────────
# Run hierarchical_rag.ipynb first to build HierarchicalRAG,
# then import it here.
import importlib, hierarchical_rag as hr_module
importlib.reload(hr_module)
from hierarchical_rag import HierarchicalRAG

hier_rag = HierarchicalRAG(rag_instance=rag)
```

Then add a wrapper to the `retrievers` dict:

```python
class HierRetrieverWrapper:
    def retrieve(self, query, top_k):
        return hier_rag.retrieve_hierarchical(query, top_k=top_k)

retrievers["7_hierarchical"] = HierRetrieverWrapper()
```